# A. Initialisation et Chargement

In [1]:
import pandas as pd
import numpy as np

# On ajoute low_memory=False pour que Pandas analyse mieux le fichier
df = pd.read_csv('../data/StockEtablissement_utf8_100000.csv', dtype=str)

# Conversion des dates (gestion des erreurs avec 'coerce' -> Si erreur, met NaT)
date_cols = ['dateCreationEtablissement', 'dateDernierTraitementEtablissement']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print("Fichier chargé avec succès sans warning !")
print(df.info()) # Pour vérifier les types

Fichier chargé avec succès sans warning !
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 54 columns):
 #   Column                                          Non-Null Count   Dtype         
---  ------                                          --------------   -----         
 0   siren                                           100000 non-null  object        
 1   nic                                             100000 non-null  object        
 2   siret                                           100000 non-null  object        
 3   statutDiffusionEtablissement                    100000 non-null  object        
 4   dateCreationEtablissement                       79316 non-null   datetime64[ns]
 5   trancheEffectifsEtablissement                   100000 non-null  object        
 6   anneeEffectifsEtablissement                     24596 non-null   object        
 7   activitePrincipaleRegistreMetiersEtablissement  1539 non-null    object       

# B. Analyse des Formats (Validité)

In [2]:
# Vérification des SIRET invalides (Longueur != 14 ou non numérique)
siret_invalides = df[~df['siret'].str.match(r'^\d{14}$')]
print(f"SIRET Invalides : {len(siret_invalides)}")

# Vérification des Codes Postaux (Doit être 5 chiffres)
cp_invalides = df[~df['codePostalEtablissement'].str.match(r'^\d{5}$', na=False)]
print(f"Codes Postaux suspects : {len(cp_invalides)}")

SIRET Invalides : 0
Codes Postaux suspects : 450


# C. Cohérence Inter-Colonnes (Logique Métier)

In [3]:
# Règle : Date Création <= Date Traitement
incoherence_temporelle = df[df['dateCreationEtablissement'] > df['dateDernierTraitementEtablissement']]

print(f"Incohérences chronologiques détectées : {len(incoherence_temporelle)}")
# Affiche un exemple pour l'analyse
print(incoherence_temporelle[['siret', 'dateCreationEtablissement', 'dateDernierTraitementEtablissement']].head())

Incohérences chronologiques détectées : 2
                siret dateCreationEtablissement  \
24702  02404133700031                2025-09-15   
37029  03578025300023                2026-01-05   

      dateDernierTraitementEtablissement  
24702                2025-09-09 22:38:09  
37029                2025-12-24 11:00:55  


# D. Analyse des Valeurs Manquantes (Complétude)

In [4]:
# Analyse croisée : Est-ce que les établissements fermés ont plus de dates manquantes ?
missing_dates = df[df['dateCreationEtablissement'].isna()]
print(missing_dates['etatAdministratifEtablissement'].value_counts(normalize=True))

F    1.0
Name: etatAdministratifEtablissement, dtype: float64
